# Job details extraction

## Library

In [1]:
from openai import OpenAI
import json
import os
from paddleocr import PaddleOCR
import requests
import numpy
from sklearn.metrics.pairwise import cosine_similarity
import re



# API function

In [ ]:
def ocr_space(file_path,api_key = "masked"):
    url = "http://api.ocr.space/parse/image"

    with open(file_path,"rb") as f:
        response = requests.post(
            url , 
            files = {"file":f},
            data ={
                "apikey":api_key,
                "language":"eng"
            }
        )
    
    result = response.json()
    text_data = result["ParsedResults"][0]["ParsedText"]

    return text_data


In [6]:
text = ocr_space("y_job_post.pdf")
print(text)

VACANCIES
Your
journey of
aspirations
begins here!
Machine Learning Engineer
8 usi
CDW
Your Friend
01 17 388388



In [9]:
with open("test1.txt","w",encoding="utf-8") as f:
    f.write(text)

# END Process

### OCR Process

#### Using paddleOCR library

In [7]:
os.environ["FLAGS_use_mkldnn"] = "0"  # Add this before importing PaddleOCR
ocr = PaddleOCR(use_textline_orientation=True, lang='en', enable_mkldnn=False)

c:\Users\User\AppData\Local\Programs\Python\Python311\Lib\site-packages\paddle\utils\cpp_extension\extension_utils.py:712: UserWarning: No ccache found. Please be aware that recompiling all source files may be required. You can download and install ccache from: https://github.com/ccache/ccache/blob/master/doc/INSTALL.md
  warnings.warn(warning_message)
Creating model: ('PP-LCNet_x1_0_doc_ori', None, None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `C:\Users\User\.paddlex\official_models\PP-LCNet_x1_0_doc_ori`.
Creating model: ('UVDoc', None, None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `C:\Users\User\.paddlex\official_models\UVDoc`.
Creating model: ('PP-LCNet_x1_0_textline_ori', None, None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `C:\Users\User\.paddlex\official_models\PP-LCNet_x1_0_textline_ori`.
Creating mode

In [8]:
result = ocr.predict('y_job_post.pdf')

### save it as txt file

In [ ]:
texts = result[0]["rec_texts"]
text_output = " ".join(texts)

with open("test2.txt", "w", encoding="utf-8") as f:
    f.write(text_output)

### Send it LLM fucntion

In [ ]:
os.environ['OPENAI_API_KEY'] = "masked"
client = OpenAI(
    base_url="https://api.groq.com/openai/v1"
)

#### Function 1

In [ ]:
# def details(text):
#     prompt = f"""
# You are an expert information extraction system.

# Extract structured information from the job post text and fill the JSON schema exactly.

# STRICT RULES:
# - Return ONLY valid JSON
# - Do NOT add extra keys
# - If value is missing, use null or empty list []
# - Do NOT explain anything
# - Do NOT use markdown

# JSON STRUCTURE:
# {{
#     "job_title":"",
#     "job_state":"",
#     "job_location":"",
#     "job_type":"",

#     "education":{{
#         "degree_status":"",
#         "required_degrees":[],
#         "preferred_degrees":[],
#         "certificates":[]
#     }},

#     "experience":{{
#         "min_experience_years":null
#     }},

#     "skills":{{
#         "technical_required":[],
#         "technical_preferred":[],
#         "soft_required":[],
#         "soft_preferred":[]
#     }},

#     "responsibilities":[],
#     "compensation": {{
#         "min_salary":null,
#         "max_salary":null
#     }},

#     "keywords":[]
# }}

# TEXT:
# {text}
# """
#     response = client.chat.completions.create(
#         model = "llama-3.3-70b-versatile",
#         messages = [{
#             "role":"user",
#             "content" : prompt
#         }],
#         temperature=0.0,
#         max_tokens = 700,
#         response_format={"type": "json_object"}
#     )

#     output = response.choices[0].message.content.strip()

#     try:
#         data =  json.loads(output)
#     except json.JSONDecodeError:
#         print("JSON Error!!!")
#         data = {}

#     return data

#### Function 2 : Claude Prompt

In [14]:
def details(text):
    prompt = f"""You are an expert HR data extraction system built to power an automated CV-to-job-post matching pipeline. Your task is to extract structured information from a raw text that is from OCR output. The text may contain minor OCR errors, irregular spacing, or formatting artifacts — interpret the content intelligently and extract all relevant fields with high precision.

EXTRACTION TASK:
Analyze the job posting image and extract all relevant fields according to the schema below. Every field you populate will directly influence how well a candidate CV is matched to this job post.

FIELD-BY-FIELD RULES:

1. job_title
   Extract the exact job title as written in the posting.

2. job_state
   Infer the seniority level from the title or description. Use ONLY one of these values:
   intern | trainee | entry-level | associate | junior | mid-level | senior | lead | principal | staff | manager | director | vp | c-level | executive
   
   MAPPING RULE:
    - Words like "Mid", "Mid-level", "Mid-level", or titles WITHOUT seniority keywords (e.g., "SE", "Software Engineer", "Data Scientist", "ML Engineer" alone) → mid-level
   If not determinable, use null.

3. job_location
   Extract the physical location (city, country, or region). If fully remote with no location, use "Remote".

4. job_type
   Use ONLY: remote | onsite | hybrid | null

5. education.degree_status
   Use "undergraduate" if the role targets current students or those without a completed degree.
   Use "graduate" if the role requires a completed bachelor's degree or higher.
   Use null if not mentioned.

6. education.required_degrees
   List degrees explicitly stated as required (e.g. "BSc in Computer Science", "MBA").

7. education.preferred_degrees
   List degrees mentioned as preferred or related, OR infer relevant degrees based on the role domain (e.g. for a Data Science role: "BSc Statistics", "BSc Mathematics", "BSc Data Science").

8. education.certificates
   List any certifications, diplomas, or professional credentials mentioned or strongly implied (e.g. "AWS Certified Solutions Architect", "Google Cloud Professional", "PMP", "CFA", "CIMA"). Infer common industry certifications if the role strongly implies them.

9. experience.min_experience_years
   Extract the minimum years of experience as an integer. Use 0 if the posting explicitly says "no experience required" or is for an internship/entry-level with no stated minimum. Use null if not mentioned.

10. skills.technical_required / technical_preferred
    Extract all technical tools, languages, frameworks, platforms, and methodologies.
    required = explicitly stated as required/must-have.
    preferred = stated as preferred/nice-to-have OR strongly implied by the role.

11. skills.soft_required / soft_preferred
    Extract interpersonal and behavioral competencies.
    required = explicitly stated.
    preferred = implied by role context or culture description.

12. responsibilities
    List all job duties, role expectations, and daily tasks. Each item should be a concise action phrase (e.g. "Design and implement machine learning models", "Collaborate with cross-functional teams").

13. compensation.min_salary / max_salary
    Extract salary figures in LKR as integers (no currency symbol, no commas). If a single figure is given, assign it to min_salary and set max_salary to null. If not mentioned, both are null.

14. keywords (CRITICAL FIELD)
    This field powers the embedding similarity matching between job posts and CVs. Extract and generate a comprehensive, semantically rich list of keywords and phrases that best represent this job posting. Include ALL of the following categories:
    - Exact and variant forms of the job title (e.g. "ML Engineer", "Machine Learning Engineer", "AI Engineer")
    - Seniority and experience level descriptors
    - All technical skills, tools, frameworks, libraries, and platforms
    - Domain and industry terms (e.g. "NLP", "computer vision", "fintech", "healthcare AI")
    - Degree fields and education keywords
    - Certification names and abbreviations
    - Key responsibility action verbs and nouns (e.g. "model deployment", "data pipeline", "A/B testing")
    - Soft skill terms relevant to this role
    - Methodologies and workflows (e.g. "agile", "scrum", "CI/CD", "MLOps")
    - Company culture or work style keywords if mentioned
    Aim for 20–40 diverse, non-redundant keywords/phrases. These keywords should maximize overlap with how a strong candidate would describe themselves in a CV.

OUTPUT RULES:
- Use null for any field where data is not present or cannot be inferred.
- All list fields must be arrays (empty array [] if nothing to extract, never null for arrays).

JSON STRUCTURE:
{{
    "job_title":"",
    "job_state":"",
    "job_location":"",
    "job_type":"",

    "education":{{
        "degree_status":"",
        "required_degrees":[],
        "preferred_degrees":[],
        "certificates":[]
    }},

    "experience":{{
        "min_experience_years":null
    }},

    "skills":{{
        "technical_required":[],
        "technical_preferred":[],
        "soft_required":[],
        "soft_preferred":[]
    }},

    "responsibilities":[],
    "compensation": {{
        "min_salary":null,
        "max_salary":null
    }},

    "keywords":[]
}}

TEXT:
{text}
"""
    response = client.chat.completions.create(
        model = "llama-3.3-70b-versatile",
        messages = [{
            "role":"user",
            "content" : prompt
        }],
        temperature=0.0,
        max_tokens = 700,
        response_format={"type": "json_object"}
    )

    output = response.choices[0].message.content.strip()

    try:
        data =  json.loads(output)
    except json.JSONDecodeError:
        print("JSON Error!!!")
        data = {}

    return data

### save as data file

In [15]:
datafile = details(text_output)
with open("jobpost_details_strongprompt.json","w",encoding="utf-8") as f:
    json.dump(datafile,f,indent=4)

# Checking similarities

In [16]:
from sentence_transformers import SentenceTransformer
model = SentenceTransformer("all-MiniLM-L6-v2")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

c:\Users\User\AppData\Local\Programs\Python\Python311\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\User\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [51]:
with open("jobpost_details_strongprompt.json", "r",encoding="utf-8") as f:
    data1 = json.load(f)

with open("one_cv_extraction.json", "r",encoding="utf-8") as f:
    data2 = json.load(f)

In [80]:
def job_education(data):
    data1 = data["education"]["required_degrees"]
    if len(data1) == 0:
        pass
    elif len(data1) >= 2:
        text = " or ".join(data1)
    else:
        text = data
    
    return text

def cv_education(data):
    cv_edu = list(data["education"]["degrees"][0].values())[0]
    return cv_edu

def job_title(data):
    return data["job_title"]
    

In [81]:
job_edu = job_education(data1)
cv_edu = cv_education(data2)
job = job_title(data1)

In [82]:
job_edu, cv_edu, job

("Bachelor's degree in Computer Science or Bachelor's degree in Data Science",
 'Bachelor of Science in Statistics (Special)',
 'Machine Learning Engineer')

In [72]:
job_emb = model.encode(job_edu, convert_to_numpy= True , normalize_embeddings=True)
cv_emb = model.encode(cv_edu,convert_to_numpy=True, normalize_embeddings=True)

#  Method 1 : embedding

In [74]:
score = cosine_similarity([job_emb], [cv_emb])[0, 0]
print(score)

0.52803886


# Method 2 :  LLM 

In [3]:
system_prompt = """You are an expert technical recruiter AI. You evaluate how appropriate a candidate's degree is for a job posting, by strictly 
following the decision rules given."""

user_prompt = """Job Title:{job_title}
Job Post Required Degree:{job_post_requirement}
Candidate's Degree (from CV):{cv_qualification}

Decision Rules (follow in this exact order, stop at first match):
STEP 1 - Exact/Equivalent Match Check:
Is the candidate's degree the same field, or an approximately equivalent field, as the job post's required degree (at the same degree level)?
-> If YES: final_score = 90

STEP 2 - Overqualification Check (only if Step 1 is NO):
Does the candidate hold a HIGHER degree level (e.g., Master's when Bachelor's is required) in the SAME or a CLOSELY RELATED field to the job post requirement?
-> If YES: final_score = 95

STEP 3 - Relevance Search (only if Steps 1 and 2 are NO):
The candidate's degree does not match the job post's required degree directly.Using the Job Title and the candidate's actual degree field, assess how 
appropriate this degree background typically is for someone working in this job title/role. Consider real-world hiring norms for this role.

Then classify into exactly one of these three outcomes:
  a) HIGH relevance to the job title -> final_score = 80
  b) LOW/SOME relevance to the job title -> final_score = 70 
  c) NOT relevant at all to the job title -> final_score = 50

### Output Format (strict JSON only, no extra text):
{{
  "final_score": <int: 90, 95, 80, 70, or 50>
}}"""

In [4]:
def details(text):
    prompt = f"""

JSON STRUCTURE:{text}
TEXT:{text}
"""
    response = client.chat.completions.create(
        model = "llama-3.3-70b-versatile",
        messages = [{
            "role":"user",
            "content" : prompt
        }],
        temperature=0.0,
        max_tokens = 20
    )

    output = response.choices[0].message.content.strip()

    try:
        data =  json.loads(output)
    except json.JSONDecodeError:
        print("JSON Error!!!")
        data = {}

    return data

In [ ]:
def score_cv_education(job_title, job_post_requirement, cv_qualification, client, model="llama-3.3-70b-versatile"):

    prompt = system_prompt + "\n\n" + user_prompt.format(
        job_title=job_title,
        job_post_requirement=job_post_requirement,
        cv_qualification=cv_qualification
    )

    messages = [{
        "role": "user",
        "content": prompt
    }]

    response = client.chat.completions.create(
        model=model,
        temperature=0,
        messages=messages
    )

    raw_output = response.choices[0].message.content

    # Defensive parsing in case Llama wraps JSON in markdown fences
    cleaned = re.sub(r'^```json\s*|\s*```$', '', raw_output.strip())

    try:
        result = json.loads(cleaned)
    except json.JSONDecodeError:
        result = {"final_score": None, "raw_output": raw_output}

    return result


In [9]:
# Example usage
if __name__ == "__main__":
    result = score_cv_education(
        job_title="Machine Learning Engineer",
        job_post_requirement="Bachelor's degree in Computer Science or Bachelor's degree in Data Science",
        cv_qualification="Bachelor of Science in Envirnment science",
        client=client
    )

    print(result)

{'final_score': 50}


# Method 3 : Gemini flash 2.0 API KEY 

In [14]:
import google.generativeai as genai
import time

In [ ]:
os.environ["GEMINI_API_KEY"] = "masked"
genai.configure(api_key = os.environ["GEMINI_API_KEY"])
model_name = "gemini-2.0-flash"

In [ ]:
SCORING_PROMPT_TEMPLATE = """
You are an expert ATS (Applicant Tracking System) evaluator for HR recruitment screening.
Your job is to score a CANDIDATE's CV against a JOB DESCRIPTION using the EXACT rubric below.
Follow the rubric strictly. Do not invent new categories. Do not skip any sub-score.

==================================================
SCORING RUBRIC (TOTAL = 100 marks)
==================================================



2) EXPERIENCE SECTION — Range 0 to 40
   Compare CV work experience (years, relevant roles, relevant industry/domain, responsibilities)
   against JD required experience (years required, role type, domain).
   - Consider: total years of relevant experience vs required years, seniority match,
     domain/industry overlap, and relevance of past responsibilities to the JD's role.
   - Score holistically across 0-40 (do not just check years — weigh relevance too).
   -> Report experience_score and a 1-2 sentence experience_justification.

3) SKILLS SECTION
   a) Technical skills — Range 0 to 35
      - If CV contains JD's "preferred_technical_skills" -> score 30-35
      - If CV contains JD's "required_technical_skills" but NOT the preferred ones -> score 15-30
      - If CV is missing JD's "required_technical_skills" -> score below 15 (use your judgment
        based on how many required skills are missing and how critical they are).
      -> Report matched_required_skills, matched_preferred_skills, missing_required_skills,
         and technical_skill_score.
   b) Soft skills — Range 0 to 5
      - Compare CV soft skills against JD soft skill expectations.
      -> Report soft_skill_score.

4) IMPACT / KEYWORD ALIGNMENT SECTION — Range 0 to 5
   and compare overall keywords with the JD's key terms.
   Score based on how strongly the CV's language and emphasis reflects what the JD is looking for.
   -> Report keyword_alignment_score.

==================================================
OUTPUT INSTRUCTIONS
==================================================
- Be strict and realistic. Do not inflate scores out of politeness.
- All scores must be numbers (not strings), and within the ranges given above
(except education_total, which may exceed 15 due to certificate bonus).
- final_total_score = education_total + experience_score + technical_skill_score + soft_skill_score + keyword_alignment_score
- Provide a short overall_summary (2-3 sentences) explaining the candidate's fit.
- Return ONLY valid JSON matching the schema.

==================================================
JOB DESCRIPTION (JSON):
==================================================
{jd_json}

==================================================
CANDIDATE CV (JSON):
==================================================
{cv_json}
"""

In [23]:
RESPONSE_SCHEMA = {
    "type": "object",
    "properties": {
        "education_score": {"type": "number"},
        "experience_score": {"type": "number"},
        "technical_skill_score": {"type": "number"},
        "soft_skill_score": {"type": "number"},
        "impact_score": {"type": "number"}
    }
}

In [25]:
def score_cv(jd_json: dict, cv_json: dict, retries: int = 3) -> dict:
    model = genai.GenerativeModel(model_name)
    prompt = SCORING_PROMPT_TEMPLATE.format(
        jd_json=json.dumps(jd_json, indent=4),
        cv_json=json.dumps(cv_json, indent=4)
    )

    config = genai.types.GenerationConfig(
        temperature=0.1,
        response_mime_type="application/json",
        response_schema=RESPONSE_SCHEMA,
    )

    for attempt in range(retries):
        try:
            response = model.generate_content(prompt, generation_config=config)
            return json.loads(response.text)
        except Exception as e:
            print(f"Attempt {attempt+1} failed: {e}")
            time.sleep(2 ** attempt)

    return {"error": "Failed after retries"}

In [2]:
# 5. Load your JD and CV JSON files
with open("jobpost_details_strongprompt.json", "r") as f:
    jd_json = json.load(f)

with open("one_cv_extraction.json", "r") as f:
    cv_json = json.load(f)


# One parts

## education parts|

In [52]:
SCORING_PROMPT_TEMPLATE = """
You are an expert ATS (Applicant Tracking System) evaluator for HR recruitment screening.
Your job is to score a CANDIDATE's CV against a JOB DESCRIPTION using the EXACT rubric below.
Follow the rubric strictly. Do not invent new categories. Do not skip any sub-score.
1) EXPERIENCE SECTION — Range 0 to 40
   Compare CV work experience (years, relevant roles, relevant industry/domain, responsibilities)
   against JD required experience (years required, role type, domain).
   - Consider: total years of relevant experience vs required years, seniority match,
     domain/industry overlap, and relevance of past responsibilities to the JD's role.
   - Score holistically across 0-40 (do not just check years — weigh relevance too).
   -> Report experience_score and a 1-2 sentence experience_justification.
==================================================
JOB DESCRIPTION (JSON):
==================================================
{jd_json}

==================================================
CANDIDATE CV (JSON):
==================================================
{cv_json}

Return ONLY a JSON object: {{"experience_score": <number>}}
No markdown, no extra text, no explanation.
"""

In [ ]:
jd = jd_json["education"]
cv = cv_json["education"]

{'degree_status': 'graduate',
 'required_degrees': ["Bachelor's degree in Computer Science",
  "Bachelor's degree in Data Science"],
 'preferred_degrees': ['related field'],
 'certificates': []}

In [49]:
cv = cv_json["education"]
cv

{'highest_qualification': 'Bachelor of Science in Statistics (Special)',
 'ol': {'year': None, 'subjects': []},
 'al': {'year': None, 'stream': None, 'subjects': [], 'z_score': None},
 'degrees': [{'degree_name': 'Bachelor of Science in Statistics (Special)',
   'university': 'University of Peradeniya',
   'gpa': 3.61}],
 'certificates': []}

In [53]:
EXPERIENCE_SCHEMA = {
    "type": "object",
    "properties": {
        "experience_score": {"type": "number"}
    }
}

In [54]:
def score_cv(jd_json: dict, cv_json: dict, retries: int = 3) -> dict:
    model = genai.GenerativeModel(model_name)
    prompt = SCORING_PROMPT_TEMPLATE.format(
        jd_json=json.dumps(jd_json, indent=4),
        cv_json=json.dumps(cv_json, indent=4)
    )

    config = genai.types.GenerationConfig(
        temperature=0.1,
        response_mime_type="application/json",
        response_schema=EXPERIENCE_SCHEMA,
    )

    for attempt in range(retries):
        try:
            response = model.generate_content(prompt, generation_config=config)
            return json.loads(response.text)
        except Exception as e:
            print(f"Attempt {attempt+1} failed: {e}")
            time.sleep(2 ** attempt)

    return {"error": "Failed after retries"}

In [ ]:
# import google.genai as genei, types

In [ ]:
# genai.Client(api_key = os.environ["GEMINI_API_KEY"])
# model_name = "gemini-2.5-flash"

In [49]:
# # Set your key directly (or via os.environ)
# genai.configure(api_key="Masked")

# try:
#     model = genai.GenerativeModel("gemini-2.5-flash")
#     response = model.generate_content("Say hello in one word.")
#     print("API key is working!")
#     print("Response:", response.text)
# except Exception as e:
#     print("API key failed or another error occurred:")
#     print(e)

In [1]:
import os
import json
import time
from google import genai
from google.genai import types
from pydantic import BaseModel

In [ ]:
os.environ["GEMINI_API_KEY"] = "MASKED"
client = genai.Client(api_key=os.environ["GEMINI_API_KEY"])
model_name = "gemini-2.5-flash"

#########################################################################################################################

# Education / Soft skill

In [3]:
class fullScore(BaseModel):
    education_score: int
    soft_skill_score: int


In [4]:
EDU_SOFT_IMPACT_PROMPT_TEMPLATE = """
You are an expert ATS (Applicant Tracking System) evaluator for HR recruitment screening.
Score ONLY the EDUCATION section, SOFT SKILL section, IMPACT / KEYWORD ALIGNMENT section of the candidate's CV against the job description.
Follow the rubric strictly. Do not invent new categories. Do not skip any step.

1) EDUCATION SECTION — Range 0 to 15 (can exceed 15 only due to certificate bonus)

   Step 1 — Identify the qualification type the JD actually requires:
      Read the JD's education/qualification requirements and classify it as ONE of:
      - "degree"  -> JD requires a specific degree/field of study (no AL/OL grades mentioned)
      - "al"      -> JD requires specific A/L (Advanced Level) subjects/grades
      - "ol"      -> JD requires specific O/L (Ordinary Level) subjects/grades
      - "none"    -> JD does not specify any formal education requirement
      Use ONLY the qualification type the JD specifies. Most JDs will specify just ONE type.
      If the JD mentions more than one type, choose the HIGHEST one mentioned
      (degree > A/L > O/L), since that is the actual hiring requirement.

   Step 2 — Score ONLY the matching section from the CV, based on qualification_type:

      a) If qualification_type = "degree":
         Compare CV degree(s) vs JD required field of study:
         - Highly related degree -> 10-15
         - Less related / loosely related degree -> 5-10
         - Not related at all -> below 5
         -> This becomes education_base_score. Do NOT score AL or OL.

      b) If qualification_type = "al":
         Compare CV A/L results vs JD required A/L subjects/grades:
         - All required subjects/grades obtained -> 10-15
         - At least one required subject/grade NOT obtained -> below 10
         -> This becomes education_base_score. Do NOT score degree or OL.

      c) If qualification_type = "ol":
         Compare CV O/L results vs JD required O/L subjects/grades:
         - All required subjects/grades obtained -> 10-15
         - At least one required subject/grade NOT obtained -> below 10
         -> This becomes education_base_score. Do NOT score degree or AL.

   Step 3 — Certificate bonus (always applies, regardless of qualification_type):
      - If CV lists certificates/courses clearly related to the JD's field or required skills,
        add +1 to education_base_score (bonus only, can push total above 15 — this is allowed).
      - If no relevant certificates, add +0.

   -> Report qualification_type (which section was used), education_base_score,
      certificate_bonus, and education_score
      (education_score = education_base_score + certificate_bonus).

2) SOFT SKILL SECTION — Range 0 to 5
Compare CV soft skills against JD soft skill.
==================================================
JOB DESCRIPTION (JSON):
==================================================
{jd_json}

==================================================
CANDIDATE CV (JSON):
==================================================
{cv_json}
"""

In [ ]:
# import tiktoken

# encoding = tiktoken.get_encoding("cl100k_base")  # GPT-4/3.5 tokenizer
# tokens = encoding.encode(full_text)

# print("Token count:", len(tokens))

Token count: 988


In [5]:
def score_education(jd_json: dict, cv_json: dict, retries: int = 3) -> dict:
    prompt = EDU_SOFT_IMPACT_PROMPT_TEMPLATE.format(
        jd_json=json.dumps(jd_json, indent=4),
        cv_json=json.dumps(cv_json, indent=4)
    )

    for attempt in range(retries):
        try:
            response = client.models.generate_content(
                model=model_name,
                contents=prompt,
                config=types.GenerateContentConfig(
                    temperature=0.0,
                    response_mime_type="application/json",
                    response_schema=fullScore,
                ),
            )
            parsed: fullScore = response.parsed
            return parsed.model_dump()
        
        except Exception as e:
            print(f"Attempt {attempt + 1} failed: {e}")
            time.sleep(30)

    return {"error": "Failed after retries"}

In [6]:
with open("jobpost_details_strongprompt.json", "r") as f:
    jd_json = json.load(f)

with open("one_cv_extraction.json", "r") as f:
    cv_json = json.load(f)

### JD Details

In [7]:
jd_selected_data = {
    "Education": jd_json["education"],
    "softskill": jd_json["skills"]["soft_required"]
}

### CV Details

In [8]:
cv_selected_data = {
    "Education": cv_json["education"],
    "softskill": cv_json["soft_skills"]
}

In [9]:

result = score_education(jd_selected_data, cv_selected_data)
print(result)

Attempt 1 failed: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-2.5-flash\nPlease retry in 19.278201249s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier_requests', 'quotaId': 'GenerateRequestsPerDayPerProjectPerModel-FreeTier', 'quotaDimensions': {'model': 'gemini-2.5-flash', 'location': 'glob

####################################################################################################################

# Tech skill

### getting keywords from cv

In [73]:
def getting_keyword(json):
    keyword = []
    keyword.extend([kw for i in range(len(json['projects'])) for kw in list(json['projects'][i].values())[2]])
    tech = json['technical_skills']
    keywords = keyword + tech
    return list(set(keywords))

### getting project details from cv

In [74]:
def getting_text_project(json):
    text = []
    for i in range(len(json["projects"])):
        data = list(json['projects'][i].values())[1]
        text.append(data)
    return text

In [ ]:
jd_selected_data_2 = {
    "tech_required_skill": jd_json["skills"]["technical_required"],
    "tech_preffered_skill": jd_json["skills"]["technical_preferred"]
}

In [ ]:
cv_selected_data_2 = {
    "technical_skill": getting_keyword(cv_json),
    "impact_details": getting_text_project(cv_json)
}

In [76]:
class skillScore(BaseModel):
    technical_score: int
    impact_score :int

In [79]:
EXP_TECH_IMPACT_PROMPT_TEMPLATE = """
You are an expert ATS (Applicant Tracking System) evaluator for HR recruitment screening.
Score ONLY the TECHNICAL SKILL SECTION and IMPACT SECTION of the candidate's CV against the job description.
Follow the rubric strictly. Do not invent new categories. Do not skip any step.

1) TECHNICAL SKILL SECTION — Range 0 to 35
   - If the MOST of technical_skill of CV contain JD's "tech_preffered" -> technical_score 30-35
   - If the MOST of technical_skill of CV contain JD's "tech_required" but NOT the preferred ones -> technical_score 23-30
   - If the LESS of technical_skill of CV contain JD's "tech_required" but NOT the preferred ones -> technical_score 15-22
   - If technical_skill of CV is missing JD's "tech_required" -> technical_score below 15

2) IMPACT SECTION — Range 0 to 5
   impact_score guide:
   5 = Almost all points are quantified with clear results and strong action verbs
   4 = Most points show results, a few are vague
   3 = Mix of results-driven and task-driven points
   2 = Mostly describes duties/tasks, little quantification
   1 = No measurable outcomes at all
   0 = No relevant content found

==================================================
JOB DESCRIPTION (JSON):
==================================================
{jd_json}

==================================================
CANDIDATE CV (JSON):
==================================================
{cv_json}
"""

In [80]:
def skill_score(jd_json: dict, cv_json: dict, retries: int = 3) -> dict:
    prompt = EXP_TECH_IMPACT_PROMPT_TEMPLATE.format(
        jd_json=json.dumps(jd_json, indent=4),
        cv_json=json.dumps(cv_json, indent=4)
    )

    for attempt in range(retries):
        try:
            response = client.models.generate_content(
                model=model_name,
                contents=prompt,
                config=types.GenerateContentConfig(
                    temperature=0.1,
                    response_mime_type="application/json",
                    response_schema=skillScore,
                ),
            )
            parsed: skillScore = response.parsed
            return parsed.model_dump()
        
        except Exception as e:
            print(f"Attempt {attempt + 1} failed: {e}")
            time.sleep(30)

    return {"error": "Failed after retries"}

In [81]:
result2 = skill_score(jd_selected_data_2, cv_selected_data_2)
print(result2)

{'technical_score': 28, 'impact_score': 4}


################################################################################################################

## Experience section

In [94]:
jd_selected_data_3 = {
    "job_role": jd_json["responsibilities"]
}

In [95]:
cv_selected_data_3 = {
    "jobs":getting_text_project(cv_json)
}

In [96]:
class ExpScore(BaseModel):
    experience_score: int

In [97]:
def score_experience_years(needed_exp, cv_exp, max_marks=15):
    if cv_exp >= needed_exp:
        return max_marks
    elif needed_exp == 0:
        return max_marks
    else:
        return round((cv_exp / needed_exp) * max_marks)

In [105]:
EXP_TEMPLATE = """
You are an expert ATS (Applicant Tracking System) evaluator for HR recruitment screening.
Score ONLY the EXPERIENCE SECTION of the candidate's CV against the job description.
Follow the rubric strictly. Do not invent new categories. Do not skip any step.

Compare the job_role below against the jobs.
Do NOT do keyword matching. Judge based on MEANING — the candidate's work may use different words but demonstrate the same skill.

experience_score how well the candidate's actual work demonstrates the required job_role, from 0 to 25:
    - 21-25 = Strong, direct evidence for almost all duties (even if worded differently)
    - 16-20 = Good evidence for most duties, some gaps
    - 11-15  = Partial overlap, several duties unaddressed
    - 6-10   = Minimal relevant evidence
    - 0-5   = No meaningful relevance

==================================================
JOB DESCRIPTION (JSON):
==================================================
{jd_json}

==================================================
CANDIDATE CV (JSON):
==================================================
{cv_json}
"""

In [106]:
def exp_score(jd_json: dict, cv_json: dict, retries: int = 3) -> dict:
    prompt = EXP_TEMPLATE.format(
        jd_json=json.dumps(jd_json, indent=4),
        cv_json=json.dumps(cv_json, indent=4)
    )

    for attempt in range(retries):
        try:
            response = client.models.generate_content(
                model=model_name,
                contents=prompt,
                config=types.GenerateContentConfig(
                    temperature=0.1,
                    response_mime_type="application/json",
                    response_schema=ExpScore,
                ),
            )
            parsed: ExpScore = response.parsed
            return parsed.model_dump()
        
        except Exception as e:
            print(f"Attempt {attempt + 1} failed: {e}")
            time.sleep(30)

    return {"error": "Failed after retries"}

In [109]:
result3 = skill_score(jd_selected_data_3, cv_selected_data_3)
print(result3)

{'experience_score': 14}


################################################################################################################################################

# Full promopt

In [125]:
PROMPT_TEMPLATE = """
You are an expert ATS (Applicant Tracking System) evaluator for HR recruitment screening.
Score ONLY the EDUCATION section, SOFT SKILL section, IMPACT / KEYWORD ALIGNMENT section of the candidate's CV against the job description.
Follow the rubric strictly. Do not invent new categories. Do not skip any step.

1) EDUCATION SECTION — Range 0 to 15 (can exceed 15 only due to certificate bonus)

   Step 1 — Identify the qualification type the JD actually requires:
      Read the JD's education/qualification requirements and classify it as ONE of:
      - "degree"  -> JD requires a specific degree/field of study (no AL/OL grades mentioned)
      - "al"      -> JD requires specific A/L (Advanced Level) subjects/grades
      - "ol"      -> JD requires specific O/L (Ordinary Level) subjects/grades
      - "none"    -> JD does not specify any formal education requirement
      Use ONLY the qualification type the JD specifies. Most JDs will specify just ONE type.
      If the JD mentions more than one type, choose the HIGHEST one mentioned
      (degree > A/L > O/L), since that is the actual hiring requirement.

   Step 2 — Score ONLY the matching section from the CV, based on qualification_type:

      a) If qualification_type = "degree":
         Compare CV degree(s) vs JD required field of study:
         - Highly related degree -> 10-15
         - Less related / loosely related degree -> 5-10
         - Not related at all -> below 5
         -> This becomes education_base_score. Do NOT score AL or OL.

      b) If qualification_type = "al":
         Compare CV A/L results vs JD required A/L subjects/grades:
         - All required subjects/grades obtained -> 10-15
         - At least one required subject/grade NOT obtained -> below 10
         -> This becomes education_base_score. Do NOT score degree or OL.

      c) If qualification_type = "ol":
         Compare CV O/L results vs JD required O/L subjects/grades:
         - All required subjects/grades obtained -> 10-15
         - At least one required subject/grade NOT obtained -> below 10
         -> This becomes education_base_score. Do NOT score degree or AL.

   Step 3 — Certificate bonus (always applies, regardless of qualification_type):
      - If CV lists certificates/courses clearly related to the JD's field or required skills,
        add +1 to education_base_score (bonus only, can push total above 15 — this is allowed).
      - If no relevant certificates, add +0.

   -> Report qualification_type (which section was used), education_base_score,
      certificate_bonus, and education_score
      (education_score = education_base_score + certificate_bonus).

2) SOFT SKILL SECTION — Range 0 to 5
Compare CV soft skills against JD soft skill.

3) TECHNICAL SKILL SECTION — Range 0 to 35
      - If the MOST of technical_skill of CV contain JD's "tech_preffered" -> technical_score 30-35
      - If the MOST of technical_skill of CV contain JD's "tech_required" but NOT the preferred ones -> technical_score 23-30
      - If the LESS of technical_skill of CV contain JD's "tech_required" but NOT the preferred ones -> technical_score 15-22
      - If technical_skill of CV is missing JD's "tech_required" -> technical_score below 15

4) IMPACT SECTION — Range 0 to 5
   impact_score guide:
   5 = Almost all points are quantified with clear results and strong action verbs
   4 = Most points show results, a few are vague
   3 = Mix of results-driven and task-driven points
   2 = Mostly describes duties/tasks, little quantification
   1 = No measurable outcomes at all
   0 = No relevant content found

5) EXPERIENCE SECTION — Range 0 to 25
   Compare the job_role below against the jobs.
   Do NOT do keyword matching. Judge based on MEANING — the candidate's work may use different words but demonstrate the same skill.

   experience_score how well the candidate's actual work demonstrates the required job_role, from 0 to 25:
   - 21-25 = Strong, direct evidence for almost all duties (even if worded differently)
   - 16-20 = Good evidence for most duties, some gaps
   - 11-15 = Partial overlap, several duties unaddressed
   - 6-10  = Minimal relevant evidence
   - 0-5   = No meaningful relevance
==================================================
JOB DESCRIPTION (JSON):
==================================================
{jd_json}

==================================================
CANDIDATE CV (JSON):
==================================================
{cv_json}
"""

## JSON of CV

In [117]:
json_cv = {
    "jobs":getting_text_project(cv_json),
    "technical_skill": getting_keyword(cv_json),
    "impact_details": getting_text_project(cv_json),
    "Education": cv_json["education"],
    "softskill": cv_json["soft_skills"]
}

## JSON of JD

In [116]:
json_jd = {
    "job_role": jd_json["responsibilities"],
    "tech_required_skill": jd_json["skills"]["technical_required"],
    "tech_preffered_skill": jd_json["skills"]["technical_preferred"],
    "Education": jd_json["education"],
    "softskill": jd_json["skills"]["soft_required"]
}

## BaseModel

In [121]:
class AllScore(BaseModel):
    experience_score: int
    technical_score: int
    impact_score :int
    education_score: int
    soft_skill_score: int
    

In [123]:
def full_score(jd_json: dict, cv_json: dict, retries: int = 3) -> dict:
    prompt = PROMPT_TEMPLATE.format(
        jd_json=json.dumps(jd_json, indent=4),
        cv_json=json.dumps(cv_json, indent=4)
    )

    for attempt in range(retries):
        try:
            response = client.models.generate_content(
                model=model_name,
                contents=prompt,
                config=types.GenerateContentConfig(
                    temperature=0.1,
                    response_mime_type="application/json",
                    response_schema=AllScore,
                ),
            )
            parsed: AllScore = response.parsed
            return parsed.model_dump()
        
        except Exception as e:
            print(f"Attempt {attempt + 1} failed: {e}")
            time.sleep(30)

    return {"error": "Failed after retries"}

In [ ]:
score = full_score(json_jd, json_jd)
print(score)

Attempt 1 failed: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-2.5-flash\nPlease retry in 58.361568225s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier_requests', 'quotaId': 'GenerateRequestsPerDayPerProjectPerModel-FreeTier', 'quotaDimensions': {'location': 'global', 'model': 'gemini-2.5-fla